# SML Dataset Process Notebook

이 노트북은 ASIS 시스템의 `scripts` 역할을 합니다.

- `packages/sml-dataset/sml_dataset`: 머신러닝 프로세스 실행에 필요한 modules
- `scripts/dataset_process.ipynb`: 실제 데이터셋 프로세스를 셀 단위로 실행하는 script

실행 범위는 데이터 생성/로드, 타겟 및 피처 선택, EDA, 전처리, pkl 저장까지입니다.

## 0. Runtime Path 설정

`scripts/` 아래에서 실행되므로 프로젝트 루트와 내부 모듈 경로를 Python path에 추가합니다.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
os.chdir(PROJECT_ROOT)

SML_DATASET_MODULES = PROJECT_ROOT / "packages" / "sml-dataset"
if str(SML_DATASET_MODULES) not in sys.path:
    sys.path.insert(0, str(SML_DATASET_MODULES))

PROJECT_ROOT

## 1. Modules Import

아래 import가 ASIS 구조에서 말하는 `scripts -> modules import`에 해당합니다.

In [ ]:
import runpy
import pandas as pd

from sml_dataset.connectors import create_connector
from sml_dataset.eda import run_eda
from sml_dataset.metadata import build_metadata
from sml_dataset.preprocessing import preprocess_dataset
from sml_dataset.significance import run_significance_tests
from sml_dataset.target_validation import validate_targets
from sml_dataset.task import TaskType
from sml_dataset.versioning import save_dataset_version

print("modules imported")

## 2. Demo Dataset 생성

운영 환경에서는 이 셀 대신 실제 DB 연결 정보를 사용하면 됩니다.
데모에서는 SQLite DB와 세 개의 테이블을 생성합니다.

In [ ]:
runpy.run_path(PROJECT_ROOT / "scripts" / "create_demo_database.py", run_name="__main__")

## 3. Task / Data Source / Target / Feature 선택

사용자가 UI에서 선택하게 될 값들을 노트북 변수로 둡니다.

- 이진분류: `binary_classification`, target 예시 `churn`
- 단일 회귀: `single_regression`, target 예시 `price`
- 멀티 회귀: `multi_regression`, target 예시 `revenue`, `conversions`

`FEATURE_COLUMNS = None`이면 target을 제외한 모든 컬럼을 feature로 사용합니다.

In [ ]:
TASK = TaskType.BINARY_CLASSIFICATION

DATA_SOURCE_TYPE = "sqlite"
DATA_SOURCE_OPTIONS = {
    "database": "data/demo_sml.db",
    "query": "SELECT * FROM customer_churn",
}

TARGET_COLUMNS = ["churn"]
FEATURE_COLUMNS = None

VERSION_NAME = "notebook_customer_churn_binary"
ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "datasets"

TASK, DATA_SOURCE_OPTIONS, TARGET_COLUMNS, FEATURE_COLUMNS

## 4. Dataset Load

connector factory를 통해 데이터 원천 타입에 맞는 loader를 생성하고 DataFrame으로 읽습니다.

In [ ]:
connector = create_connector(DATA_SOURCE_TYPE, DATA_SOURCE_OPTIONS)
raw_df = connector.load()

raw_df.shape

In [ ]:
raw_df.head()

## 5. Metadata 생성

사용자에게 보여줄 컬럼 타입, 결측치, 고유값 수, preview를 생성합니다.

In [ ]:
metadata = build_metadata(raw_df, preview_rows=5)
pd.DataFrame(metadata["columns"])

## 6. Target 검증 및 Feature 선택

태스크별 target 규칙을 검증한 뒤 feature 컬럼을 확정합니다.

In [ ]:
validate_targets(raw_df, TASK, TARGET_COLUMNS)

if FEATURE_COLUMNS is None:
    FEATURE_COLUMNS = [column for column in raw_df.columns if column not in TARGET_COLUMNS]

selected_df = raw_df[FEATURE_COLUMNS + TARGET_COLUMNS].copy()

print("target validation passed")
print("features:", FEATURE_COLUMNS)
print("targets:", TARGET_COLUMNS)
selected_df.head()

## 7. EDA 실행

EDA 결과는 요약 통계와 화면 렌더링용 `chart_specs`를 포함합니다.

In [ ]:
eda = run_eda(selected_df, TASK, TARGET_COLUMNS)

display(pd.Series(eda["missing_by_column"], name="missing_count").to_frame())
display(pd.DataFrame(eda["chart_specs"]).head(20))

## 8. 유의성 검증

- 이진분류: WoE/IV
- 회귀: Pearson correlation, Mutual Information, VIF

In [ ]:
significance = run_significance_tests(selected_df, TASK, TARGET_COLUMNS)

if significance.get("method") == "woe_iv":
    display(pd.DataFrame(significance["iv_by_feature"]).T)
    first_feature = next(iter(significance["woe_bins"]), None)
    if first_feature:
        print("WoE bins:", first_feature)
        display(pd.DataFrame(significance["woe_bins"][first_feature]))
else:
    for target, result in significance["pearson"].items():
        print("Pearson:", target)
        display(pd.DataFrame(result).T)
    for target, result in significance["mutual_information"].items():
        print("Mutual information:", target)
        display(pd.Series(result, name="mi").to_frame())
    display(pd.DataFrame(significance["vif"]).T)

## 9. 전처리 설정

인코딩과 스케일링을 포함한 전처리 옵션입니다.
이 값들이 실제 시스템에서는 사용자가 UI에서 선택하는 값이 됩니다.

In [ ]:
PREPROCESSING_OPTIONS = {
    "drop_duplicates": True,
    "missing": {
        "numeric": "median",
        "categorical": "most_frequent",
    },
    "outliers": {
        "method": "iqr_remove",
        "factor": 1.5,
    },
    "encoding": {
        "method": "onehot",
    },
    "scaling": {
        "method": "standard",
    },
}

PREPROCESSING_OPTIONS

## 10. 전처리 실행

target을 분리한 뒤 feature에만 결측치 처리, 이상치 처리, 인코딩, 스케일링을 적용합니다.

In [ ]:
preprocess_result = preprocess_dataset(selected_df, TARGET_COLUMNS, PREPROCESSING_OPTIONS)

display(preprocess_result.report)
display(preprocess_result.features.head())
display(preprocess_result.targets.head())

## 11. Dataset Version pkl 저장

전처리된 feature, target, metadata, preprocessing report, preprocessor를 하나의 pkl로 저장합니다.
이 pkl은 이후 모델링 script의 입력으로 사용됩니다.

In [ ]:
version_info = save_dataset_version(
    name=VERSION_NAME,
    artifact_dir=ARTIFACT_DIR,
    features=preprocess_result.features,
    targets=preprocess_result.targets,
    metadata=metadata,
    preprocessing_report=preprocess_result.report,
    preprocessor=preprocess_result.preprocessor,
)

version_info

## 12. 저장 결과 확인

In [ ]:
import joblib

payload = joblib.load(version_info.path)
payload.keys(), payload["features"].shape, payload["targets"].shape

## 다른 태스크로 실행하는 방법

3번 셀의 값을 아래처럼 바꾸면 같은 script 흐름을 다른 태스크에도 적용할 수 있습니다.

단일 회귀:

```python
TASK = TaskType.SINGLE_REGRESSION
DATA_SOURCE_OPTIONS = {
    "database": "data/demo_sml.db",
    "query": "SELECT * FROM house_prices",
}
TARGET_COLUMNS = ["price"]
VERSION_NAME = "notebook_house_price_single_regression"
```

멀티 회귀:

```python
TASK = TaskType.MULTI_REGRESSION
DATA_SOURCE_OPTIONS = {
    "database": "data/demo_sml.db",
    "query": "SELECT * FROM ad_campaign",
}
TARGET_COLUMNS = ["revenue", "conversions"]
VERSION_NAME = "notebook_ad_campaign_multi_regression"
```